In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass
import math

# 1. Define reactor properties

In [ ]:
@dataclass(frozen=True)
class Material1D():
        # Inherent properties
        sigma_s:float
        sigma_c:float
        sigma_f:float 
        nu:float

        def __post_init__(self) -> None:
            values = (self.sigma_s, self.sigma_c, self.sigma_f, self.nu)

            if not all(math.isfinite(value) for value in values):
                raise ValueError("Material coefficients must be finite")

            if any(value < 0.0 for value in values):
                raise ValueError("Material coefficients must be non-negative")

            if self.sigma_t <= 0.0:
                raise ValueError("sigma_t must be strictly positive")
        #Computed properties
        @property
        def sigma_t(self) -> float:
            return self.sigma_f + self.sigma_c + self.sigma_s
        @property
        def sigma_a(self) -> float:
            return self.sigma_c + self.sigma_s
        @property
        def sigma_p(self)->float:
            return self.sigma_c + self.nu * self.sigma_f
        @property
        def diffusion_coefficient(self)->float:
             return 1.0/(3.0*self.sigma_t)


In [ ]:
@dataclass(frozen=True)
class Reactor1D():

    half_width:float
    core_half_width: float
    speed: float
    core: Material1D
    reflector: Material1D

    def __post_init__(self)->None:
        assert self.half_width >= self.core_half_width
        assert self.speed > 0


In [ ]:
class UniformMesh1D():
    def __init__(self, reactor:Reactor1D, n_cells:int):
        # Mesh spatial params
        self.n_cells = np.floor(n_cells)
        self.h = 2*reactor.half_width / self.n_cells
        # Material mesh arrays
        self.faces:np.ndarray = np.arange(-reactor.half_width,reactor.half_width,self.h)
        self.centers:np.ndarray = np.arange(-reactor.half_width+self.h/2,reactor.half_width-self.h/2,self.h)
        #Check mesh sizes
        assert self.centers.size == self.n_cells - 1
        assert self.faces.size == self.n_cells
        #Check interface is on a face 
        distance = reactor.half_width + reactor.core_half_width
        if not np.isclose(distance / self.h, round(distance / self.h)):
            raise ValueError("Heart-reflector interface should lie on a mesh face")
        #Masque de coeur pour définir les materiaux 
        self.core_mask = np.abs(self.centers) < reactor.core_half_width
    
        self.sigma_s = np.where(self.core_mask,reactor.core.sigma_s,reactor.reflector.sigma_s)
        self.sigma_f = np.where(self.core_mask,reactor.core.sigma_f,reactor.reflector.sigma_f)
        self.sigma_c = np.where(self.core_mask,reactor.core.sigma_c,reactor.reflector.sigma_c)
        self.nu = np.where(self.core_mask,reactor.core.nu,reactor.reflector.nu)
        
        self.sigma_p = np.where(self.core_mask,reactor.core.sigma_p,reactor.reflector.sigma_p)
        self.sigma_t = np.where(self.core_mask,reactor.core.sigma_t,reactor.reflector.sigma_t)
        self.diffusion_coeff = np.where(self.core_mask,reactor.core.diffusion_coefficient,reactor.reflector.diffusion_coefficient)

In [ ]:
L = 10 # diameter of reactor   
a = 4 # fission heart diameter
J = 20000 # total cells number

Ss = 1
Sf = 1.5
Sc = 2
nu = 2.5

In [ ]:
Core = Material1D(sigma_s=Ss,sigma_f=Sf,sigma_c=Sc,nu=nu)
Reflector = Material1D(sigma_s=Ss,sigma_c=Sc,sigma_f=0,nu=0)
Reactor = Reactor1D(half_width=L,core_half_width=a,speed=1,core=Core,reflector=Reflector)
Mesh = UniformMesh1D(reactor=Reactor, n_cells=J)

In [ ]:
plt.figure(figsize=(14,6),dpi=720)
plt.plot(Mesh.centers, Mesh.sigma_f)
plt.plot(Mesh.centers, Mesh.sigma_p)
plt.plot(Mesh.centers, Mesh.diffusion_coeff)
plt.show()

In [ ]:
from numpy.polynomial import legendre

@dataclass(frozen=True)
class SNQuadrature:
    nodes: np.ndarray
    weights: np.ndarray

    @property
    def order(self) -> int:
        return len(self.nodes)

    @classmethod
    def gauss_legendre(cls, order: int) -> "SNQuadrature":
        if not isinstance(order, int) or order <= 0 or order % 2 != 0:
            raise ValueError("order must be a positive even integer")

        nodes, weights = np.polynomial.legendre.leggauss(order)
        return cls(nodes=nodes, weights=weights)

In [ ]:
quadrature = SNQuadrature.gauss_legendre(16)
quadrature.weights